# Stage 30 — Fixed LevelAttentionModule (Exp B)

**Hypothesis (RQ11):** Two architectural changes in `LevelAttentionModule.forward()` break
the L2 feedback loop and produce stable, seed-independent level discovery:

1. **Feature detach** — `x = cat(pooled).detach()` — attention is a pure reader of features,
   not a writer. L2 cannot self-reinforce via gradient.
2. **Temperature annealing** — `softmax(logits / T)`, T: 5→1 over 40 epochs after unfreeze —
   prevents init-dependent early commitment.

**Stability test:** 3 seeds (42, 0, 123). Pass = all converge to same dominant levels.

**Notebook contains:** Stage 1 training (3 seeds) → attention evolution plots → stability
table → proto quality per seed → RQ11 verdict.

## 0. Config

In [1]:
import sys, os

_root = (
    os.path.dirname(os.getcwd())
    if os.path.basename(os.getcwd()) == "notebooks"
    else os.getcwd()
)
os.chdir(_root)
sys.path.insert(0, _root)
os.environ.setdefault("PYTORCH_MPS_HIGH_WATERMARK_RATIO", "0.0")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# ── Seeds to test ─────────────────────────────────────────────────────────────
SEEDS = [42, 0, 123]

# ── Model ─────────────────────────────────────────────────────────────────────
PROTO_LEVELS = [1, 2, 3, 4]
USE_LEVEL_ATTENTION = True
MODALITY = "ct"

# ── Temperature annealing ─────────────────────────────────────────────────────
T_START = 5.0  # softmax near-uniform at start
T_END = 1.0  # standard softmax at end
ANNEAL_EP = 40  # epochs after attention unfreeze to reach T=1

# ── Training ──────────────────────────────────────────────────────────────────
LAMBDA_DIV = 0.001
LAMBDA_PUSH = 0.5
LAMBDA_PULL = 0.25
BATCH_SIZE = 16
LR = 3e-4
WEIGHT_DECAY = 1e-5
PHASE_A_END = 20
ATTN_WARMUP_EP = 10  # attention MLP frozen for this many epochs after Phase A
TOTAL_EPOCHS = 60  # stop early — just need hierarchy confirmation
VAL_EVERY = 5
PROJ_INTERVAL = 10
HIERARCHY_THRESHOLD = 0.05
HIERARCHY_PATIENCE = 3

DATA_DIR = "data/pack/processed_data"
CKPT_DIR = "checkpoints"
LOG_DIR = "results/v7"

import pathlib

pathlib.Path(LOG_DIR).mkdir(parents=True, exist_ok=True)


def ckpt_path(seed):
    return f"{CKPT_DIR}/proto_seg_ct_l1234_attn_fixed_seed{seed}.pth"


def log_path(seed):
    return f"{LOG_DIR}/attn_evolution_fixed_seed{seed}.csv"


print(f"Seeds         : {SEEDS}")
print(f"Temperature   : {T_START}→{T_END} over {ANNEAL_EP} epochs post-unfreeze")
print(f"Total epochs  : {TOTAL_EPOCHS}  (stop at hierarchy confirmation)")
print(f"Feature detach: enabled (in LevelAttentionModule.forward)")

Seeds         : [42, 0, 123]
Temperature   : 5.0→1.0 over 40 epochs post-unfreeze
Total epochs  : 60  (stop at hierarchy confirmation)
Feature detach: enabled (in LevelAttentionModule.forward)


## 1. Imports & Device

In [2]:
import csv, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.data.mmwhs_dataset import (
    MMWHSSliceDataset,
    make_dataloaders,
    LABEL_NAMES,
    NUM_CLASSES,
)
from src.models.proto_seg_net import ProtoSegNet
from src.models.prototype_layer import PrototypeProjection
from src.losses.segmentation import SegmentationLoss
from src.losses.diversity_loss import ProtoSegLoss
from src.metrics.dice import dice_per_class, mean_foreground_dice
from src.metrics.proto_quality import (
    compute_purity,
    compute_per_level_ap,
    compute_level_dominance,
    compute_effective_quality,
    compute_compactness,
)

DEVICE = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)
FG_NAMES = [LABEL_NAMES[c] for c in range(1, NUM_CLASSES)]
print(f"Device: {DEVICE}")

Device: mps


## 2. Data & Class Weights

In [3]:
loaders = make_dataloaders(DATA_DIR, MODALITY, batch_size=BATCH_SIZE)
print(
    f"Train: {len(loaders['train'].dataset)}  "
    f"Val: {len(loaders['val'].dataset)}  "
    f"Test: {len(loaders['test'].dataset)}"
)
class_weights = torch.load(f"data/class_weights_{MODALITY}.pt", weights_only=True)
print(f"Class weights: {class_weights.numpy().round(3)}")

Train: 3389  Val: 382  Test: 484
Class weights: [0.021 0.819 1.197 1.066 1.192 0.795 1.486 1.423]


## 3. Training Loop (per seed)

Runs Stage 1 three times with different seeds. Each run:
- Phase A (ep 1–20): encoder + decoder, prototypes + attention frozen
- Phase B (ep 21+): all params; attention MLP warmup for first 10 epochs
- Temperature anneals T: 5→1 over `ANNEAL_EP` epochs post-unfreeze
- Stops when `w_L1+L2 < 0.05` for `HIERARCHY_PATIENCE` consecutive val epochs

In [ ]:
def train_fixed_stage1(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = ProtoSegNet(
        n_classes=NUM_CLASSES,
        proto_levels=PROTO_LEVELS,
        use_level_attention=USE_LEVEL_ATTENTION,
    ).to(DEVICE)

    seg_loss = SegmentationLoss(
        class_weights=class_weights.to(DEVICE), n_classes=NUM_CLASSES
    )
    criterion = ProtoSegLoss(
        seg_loss=seg_loss,
        lambda_div=LAMBDA_DIV,
        lambda_push=LAMBDA_PUSH,
        lambda_pull=LAMBDA_PULL,
    )

    # ── Helpers ───────────────────────────────────────────────────────────────
    @torch.no_grad()
    def validate():
        model.eval()
        ll, lb = [], []
        for b in loaders["val"]:
            logits, _ = model(b["image"].to(DEVICE))
            ll.append(logits.cpu())
            lb.append(b["label"])
        model.train()
        return mean_foreground_dice(dice_per_class(torch.cat(ll), torch.cat(lb)))

    @torch.no_grad()
    def mean_attn():
        model.eval()
        ws = []
        for b in loaders["val"]:
            ws.append(model.get_attention_weights(b["image"].to(DEVICE)).cpu())
        model.train()
        return torch.cat(ws).mean(0).numpy()

    def run_proj(save_path):
        proj_ds = MMWHSSliceDataset(
            DATA_DIR, MODALITY, "train", augment=False, preload=True
        )
        proj_loader = torch.utils.data.DataLoader(proj_ds, batch_size=32, shuffle=False)
        wrapped = [(b["image"], b["label"]) for b in proj_loader]
        PrototypeProjection(
            encoder=model.encoder, proto_layers=model.proto_layers_dict(), device="cpu"
        ).project(wrapped, save_path=save_path)
        model.to(DEVICE)
        ckpt = torch.load(save_path, weights_only=True)
        for level, pd_ in ckpt["proto_state"].items():
            model.proto_layers[str(level)].prototypes.data.copy_(pd_)

    def compute_temperature(epoch):
        """Linear anneal T_START→T_END over ANNEAL_EP epochs after attention unfreeze."""
        attn_unfreeze_ep = PHASE_A_END + ATTN_WARMUP_EP
        if epoch <= attn_unfreeze_ep:
            return T_START
        progress = min(1.0, (epoch - attn_unfreeze_ep) / ANNEAL_EP)
        return T_START + progress * (T_END - T_START)

    def set_phase(epoch, optimizer):
        if epoch <= PHASE_A_END:
            model.unfreeze_all()
            model.freeze_prototypes()
            phase = "A"
        else:
            model.unfreeze_all()
            if epoch <= PHASE_A_END + ATTN_WARMUP_EP:
                for p in model.level_attention.parameters():
                    p.requires_grad_(False)
            phase = "B"
        optimizer.param_groups[0]["params"] = [
            p for p in model.parameters() if p.requires_grad
        ]
        return phase

    # ── Init ──────────────────────────────────────────────────────────────────
    model.freeze_prototypes()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

    lpath = log_path(seed)
    cpath = ckpt_path(seed)
    proj_path = f"{CKPT_DIR}/projected_proto_ct_fixed_seed{seed}.pt"

    attn_file = open(lpath, "w", newline="")
    attn_writer = csv.DictWriter(
        attn_file,
        fieldnames=["epoch", "phase", "temperature"]
        + [f"w_L{l}" for l in PROTO_LEVELS],
    )
    attn_writer.writeheader()

    best_val, hier_ok, hier_epoch = 0.0, 0, None
    current_phase = "A"

    print(f"\n{'=' * 60}")
    print(f"Seed {seed}  |  detach=True  |  T: {T_START}→{T_END} over {ANNEAL_EP} ep")
    print(f"{'=' * 60}")

    for epoch in range(1, TOTAL_EPOCHS + 1):
        new_phase = set_phase(epoch, optimizer)
        if new_phase != current_phase:
            current_phase = new_phase
            print(
                f"\n→ Phase B (ep {PHASE_A_END + 1}+)  attn warmup until ep {PHASE_A_END + ATTN_WARMUP_EP}"
            )
            best_val, hier_ok = 0.0, 0

        # Set temperature on the attention module
        T = compute_temperature(epoch)
        if model.level_attention is not None:
            model.level_attention.temperature = T

        if (
            current_phase == "B"
            and epoch > PHASE_A_END + 1
            and (epoch - PHASE_A_END) % PROJ_INTERVAL == 0
        ):
            run_proj(proj_path)

        # Train
        t0 = time.time()
        model.train()
        totals = dict(loss=0, n=0)
        for batch in loaders["train"]:
            imgs = batch["image"].to(DEVICE)
            lbls = batch["label"].to(DEVICE)
            optimizer.zero_grad()
            logits, hm = model(imgs)
            out = (
                seg_loss(logits, lbls)
                if current_phase == "A"
                else criterion(logits, lbls, hm)
            )
            out["loss"].backward()
            optimizer.step()
            totals["loss"] += out["loss"].item()
            totals["n"] += 1
        scheduler.step()
        epoch_time = time.time() - t0

        if epoch % VAL_EVERY == 0 or epoch == TOTAL_EPOCHS:
            val_mean = validate()
            attn_w = mean_attn()
            w_low = float(attn_w[0] + attn_w[1])  # w_L1 + w_L2

            row = {"epoch": epoch, "phase": current_phase, "temperature": round(T, 3)}
            for i, l in enumerate(PROTO_LEVELS):
                row[f"w_L{l}"] = round(float(attn_w[i]), 4)
            attn_writer.writerow(row)
            attn_file.flush()

            attn_active = current_phase == "B" and epoch > PHASE_A_END + ATTN_WARMUP_EP
            if attn_active and w_low < HIERARCHY_THRESHOLD:
                hier_ok += 1
            else:
                hier_ok = 0

            if val_mean > best_val:
                best_val = val_mean
                torch.save(
                    {
                        "epoch": epoch,
                        "model_state_dict": model.state_dict(),
                        "best_val_dice": best_val,
                        "proto_levels": model.proto_levels,
                        "use_level_attention": True,
                        "seed": seed,
                        "class_weights": class_weights,
                        "lambda_div": LAMBDA_DIV,
                        "lambda_push": LAMBDA_PUSH,
                        "lambda_pull": LAMBDA_PULL,
                        "single_scale": model.single_scale,
                        "no_soft_mask": model.no_soft_mask,
                        "hard_mask": model.hard_mask,
                        "mask_quantile": model.mask_quantile,
                        "hard_mask_active": model.hard_mask_active,
                    },
                    cpath,
                )

            hier_str = f"  [hier {hier_ok}/{HIERARCHY_PATIENCE}]" if attn_active else ""
            print(
                f"  [{current_phase}] Ep {epoch:3d} | "
                f"loss={totals['loss'] / totals['n']:.4f} | val={val_mean:.4f} | "
                f"T={T:.2f} | w=[{','.join(f'{v:.2f}' for v in attn_w)}] "
                f"w_L1+L2={w_low:.3f}{hier_str} | {epoch_time:.1f}s",
                flush=True,
            )

            if hier_ok >= HIERARCHY_PATIENCE:
                hier_epoch = epoch
                # Save final checkpoint at hierarchy confirmation
                torch.save(
                    {
                        "epoch": epoch,
                        "model_state_dict": model.state_dict(),
                        "best_val_dice": best_val,
                        "proto_levels": model.proto_levels,
                        "use_level_attention": True,
                        "seed": seed,
                        "class_weights": class_weights,
                        "lambda_div": LAMBDA_DIV,
                        "lambda_push": LAMBDA_PUSH,
                        "lambda_pull": LAMBDA_PULL,
                        "single_scale": model.single_scale,
                        "no_soft_mask": model.no_soft_mask,
                        "hard_mask": model.hard_mask,
                        "mask_quantile": model.mask_quantile,
                        "hard_mask_active": model.hard_mask_active,
                    },
                    cpath,
                )
                print(
                    f"\n✅ Hierarchy confirmed at epoch {epoch}  (w_L1+L2={w_low:.4f})"
                )
                break

    attn_file.close()
    if hier_epoch is None:
        print(f"⚠️  Max epochs reached without confirmation (seed={seed})")
        hier_epoch = -1

    # Read final attention row
    df = pd.read_csv(lpath)
    final = df.iloc[-1]
    return {
        "seed": seed,
        "hier_epoch": hier_epoch,
        "best_val": best_val,
        "final_attn": {f"w_L{l}": float(final[f"w_L{l}"]) for l in PROTO_LEVELS},
        "ckpt": cpath,
        "log": lpath,
    }


# ── Run all seeds ─────────────────────────────────────────────────────────────
seed_results = {}
for seed in SEEDS:
    seed_results[seed] = train_fixed_stage1(seed)

print("\n" + "=" * 60)
print("Summary")
print("=" * 60)
print(
    f"{'Seed':>6} {'Hier ep':>8} {'Best val':>9}  "
    + "  ".join(f"w_L{l}" for l in PROTO_LEVELS)
)
for seed, r in seed_results.items():
    w_str = "  ".join(f"{r['final_attn'][f'w_L{l}']:>6.3f}" for l in PROTO_LEVELS)
    print(f"  {seed:>4}   {r['hier_epoch']:>7}   {r['best_val']:>8.4f}   {w_str}")


Seed 42  |  detach=True  |  T: 5.0→1.0 over 40 ep
  [A] Ep   5 | loss=0.2563 | val=0.7335 | T=5.00 | w=[0.26,0.26,0.25,0.23] w_L1+L2=0.515 | 45.3s
  [A] Ep  10 | loss=0.1214 | val=0.7429 | T=5.00 | w=[0.26,0.26,0.25,0.23] w_L1+L2=0.515 | 45.7s
  [A] Ep  15 | loss=0.0929 | val=0.8014 | T=5.00 | w=[0.26,0.26,0.26,0.23] w_L1+L2=0.513 | 45.3s
  [A] Ep  20 | loss=0.0770 | val=0.8091 | T=5.00 | w=[0.26,0.26,0.25,0.23] w_L1+L2=0.515 | 44.8s

→ Phase B (ep 21+)  attn warmup until ep 30
  [B] Ep  25 | loss=6.8221 | val=0.7016 | T=5.00 | w=[0.25,0.26,0.25,0.24] w_L1+L2=0.513 | 61.9s
Projected prototypes saved → checkpoints/projected_proto_ct_fixed_seed42.pt
  [B] Ep  30 | loss=2.6379 | val=0.7101 | T=5.00 | w=[0.25,0.26,0.25,0.23] w_L1+L2=0.513 | 61.3s
  [B] Ep  35 | loss=1.4335 | val=0.8081 | T=4.50 | w=[0.09,0.35,0.29,0.27] w_L1+L2=0.436  [hier 0/3] | 61.1s
Projected prototypes saved → checkpoints/projected_proto_ct_fixed_seed42.pt
  [B] Ep  40 | loss=1.1074 | val=0.7998 | T=4.00 | w=[0.05,0.

## 4. Attention Evolution Plots

In [ ]:
LEVEL_COLORS = {1: "#d62728", 2: "#ff7f0e", 3: "#2ca02c", 4: "#1f77b4"}

fig, axes = plt.subplots(1, len(SEEDS), figsize=(5 * len(SEEDS), 4), sharey=True)

for ax, seed in zip(axes, SEEDS):
    df = pd.read_csv(log_path(seed))
    r = seed_results[seed]
    for l in PROTO_LEVELS:
        ax.plot(
            df["epoch"], df[f"w_L{l}"], lw=1.8, color=LEVEL_COLORS[l], label=f"L{l}"
        )
    ax.axhline(
        HIERARCHY_THRESHOLD,
        ls=":",
        lw=1,
        color="gray",
        label=f"threshold={HIERARCHY_THRESHOLD}",
    )
    if r["hier_epoch"] > 0:
        ax.axvline(
            r["hier_epoch"],
            ls="--",
            lw=1.2,
            color="black",
            label=f"confirmed ep{r['hier_epoch']}",
        )
    ax.set_title(f"Seed {seed}  (confirmed ep {r['hier_epoch']})")
    ax.set_xlabel("Epoch")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Attention weight")
plt.suptitle(
    "Attention Weight Evolution — Fixed Stage 1 (detach + temperature)", fontsize=11
)
plt.tight_layout()
plt.savefig(f"{LOG_DIR}/attn_evolution_fixed_seeds.png", dpi=150, bbox_inches="tight")
plt.show()

# Also plot temperature schedule on a separate axis
fig2, ax2 = plt.subplots(figsize=(6, 3))
sample_df = pd.read_csv(log_path(SEEDS[0]))
ax2.plot(sample_df["epoch"], sample_df["temperature"], lw=2, color="purple")
ax2.axhline(T_END, ls=":", lw=1, color="gray", label=f"T_END={T_END}")
ax2.set(title="Temperature Schedule", xlabel="Epoch", ylabel="Temperature T")
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Stability Table

In [ ]:
# Compare with v6 noent (unfixed) reference: seed=42 → w_L2=0.30, w_L4=0.63
V6_NOENT_REF = {"w_L1": 0.007, "w_L2": 0.298, "w_L3": 0.063, "w_L4": 0.632}

print("Stability Comparison — Fixed vs Unfixed Stage 1")
print()
print(
    f"  {'Model':<30} {'Hier ep':>8} {'w_L1':>6} {'w_L2':>6} {'w_L3':>6} {'w_L4':>6}  Stable?"
)
print("  " + "-" * 72)
print(
    f"  {'Unfixed (v6, seed=42)':<30} {'45':>8} "
    + "  ".join(f"{V6_NOENT_REF[f'w_L{l}']:>6.3f}" for l in PROTO_LEVELS)
    + "  ❌ L2 locked at 30%"
)

stable_flags = []
for seed, r in seed_results.items():
    w = r["final_attn"]
    w_low = w["w_L1"] + w["w_L2"]
    w_high = w["w_L3"] + w["w_L4"]
    stable = w_low < HIERARCHY_THRESHOLD and w_high > 0.90
    stable_flags.append(stable)
    flag = "✅ L1+L2 suppressed" if stable else f"❌ w_L1+L2={w_low:.3f}"
    print(
        f"  {'Fixed seed=' + str(seed):<30} {r['hier_epoch']:>8} "
        + "  ".join(f"{w[f'w_L{l}']:>6.3f}" for l in PROTO_LEVELS)
        + f"  {flag}"
    )

all_stable = all(stable_flags)
print()
print(
    f"RQ11 stability: {'✅ ALL seeds stable' if all_stable else '❌ some seeds unstable'}"
)

# Save stability table
rows = []
for seed, r in seed_results.items():
    w = r["final_attn"]
    rows.append(
        {
            "seed": seed,
            "hier_epoch": r["hier_epoch"],
            "best_val": r["best_val"],
            **{f"w_L{l}": w[f"w_L{l}"] for l in PROTO_LEVELS},
            "w_L1_L2": w["w_L1"] + w["w_L2"],
            "stable": all_stable,
        }
    )
pd.DataFrame(rows).to_csv(f"{LOG_DIR}/stability_fixed_seeds.csv", index=False)
print(f"Saved: {LOG_DIR}/stability_fixed_seeds.csv")

## 6. Proto Quality per Seed

Using the best-val checkpoint from each seed. Computes effective quality
to check whether the fixed encoder produces better prototype selectivity.

In [ ]:
train_ds = MMWHSSliceDataset(DATA_DIR, MODALITY, "train", augment=False, preload=True)
test_ds = MMWHSSliceDataset(DATA_DIR, MODALITY, "test", augment=False, preload=True)
train_loader_pq = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=False)
test_loader_pq = torch.utils.data.DataLoader(test_ds, batch_size=32, shuffle=False)

quality_rows = []
for seed, r in seed_results.items():
    ckpt = torch.load(r["ckpt"], map_location=DEVICE, weights_only=False)
    m = ProtoSegNet(
        n_classes=NUM_CLASSES,
        proto_levels=ckpt["proto_levels"],
        use_level_attention=ckpt.get("use_level_attention", True),
    ).to(DEVICE)
    m.load_state_dict(ckpt["model_state_dict"])
    m.eval()
    print(
        f"\nSeed {seed}  ep={ckpt['epoch']}  best_val={ckpt['best_val_dice']:.4f}",
        flush=True,
    )

    purity_df = compute_purity(m, train_loader_pq)
    compact_df = compute_compactness(m, test_loader_pq)
    ap_df = compute_per_level_ap(m, test_loader_pq)
    dom_df = compute_level_dominance(m, test_loader_pq)
    eff = compute_effective_quality(purity_df, ap_df, compact_df, dom_df)

    purity_l = purity_df.groupby("level")["purity"].mean()
    ap_l = ap_df.groupby("level")["ap"].mean()

    print(f"  {'Level':<6} {'Purity':>7} {'AP':>7} {'Dominance':>10}")
    for l in PROTO_LEVELS:
        dom = (
            dom_df[f"frac_l{l}"].values[0]
            if f"frac_l{l}" in dom_df.columns
            else float("nan")
        )
        print(
            f"  L{l:<5} {purity_l.get(l, float('nan')):>7.3f} {ap_l.get(l, float('nan')):>7.3f} {dom:>9.1%}"
        )
    print(
        f"  effective_purity={float(eff['effective_purity'].values[0]):.3f}  "
        f"effective_ap={float(eff['effective_ap'].values[0]):.3f}"
    )

    quality_rows.append(
        {
            "seed": seed,
            "hier_epoch": r["hier_epoch"],
            "best_val": r["best_val"],
            **{
                f"purity_L{l}": float(purity_l.get(l, float("nan")))
                for l in PROTO_LEVELS
            },
            **{f"ap_L{l}": float(ap_l.get(l, float("nan"))) for l in PROTO_LEVELS},
            **{
                f"dom_L{l}": float(dom_df[f"frac_l{l}"].values[0])
                if f"frac_l{l}" in dom_df.columns
                else float("nan")
                for l in PROTO_LEVELS
            },
            "effective_purity": float(eff["effective_purity"].values[0]),
            "effective_ap": float(eff["effective_ap"].values[0]),
        }
    )

df_q = pd.DataFrame(quality_rows)
df_q.to_csv(f"{LOG_DIR}/proto_quality_fixed_seeds.csv", index=False)
print(f"\nSaved: {LOG_DIR}/proto_quality_fixed_seeds.csv")

## 7. RQ11 Verdict

In [ ]:
print("RQ11: Does fixing LevelAttentionModule (detach + temperature) produce")
print("      stable, seed-independent level discovery?")
print()

all_confirmed = all(r["hier_epoch"] > 0 for r in seed_results.values())
all_l2_below = all(
    r["final_attn"]["w_L1"] + r["final_attn"]["w_L2"] < HIERARCHY_THRESHOLD
    for r in seed_results.values()
)
all_l34_above = all(
    r["final_attn"]["w_L3"] + r["final_attn"]["w_L4"] > 0.90
    for r in seed_results.values()
)

criteria = [
    ("All seeds reach hierarchy confirmation", all_confirmed),
    (f"All seeds: w_L1+L2 < {HIERARCHY_THRESHOLD}", all_l2_below),
    ("All seeds: w_L3+L4 > 0.90", all_l34_above),
]
passed = sum(1 for _, ok in criteria if ok)
for desc, ok in criteria:
    print(f"  {'✅' if ok else '❌'}  {desc}")

print()
if passed == len(criteria):
    print("Verdict: ✅ STABLE — all seeds converge to same hierarchy.")
    print("→ Proceed to Exp C (full automated pipeline).")
    print(f"→ Use seed=42 checkpoint as Stage 1 encoder for Exp C: {ckpt_path(42)}")
elif all_confirmed and not all_l2_below:
    print(
        "Verdict: ⚠️  PARTIAL — hierarchy confirmed but L2 still non-negligible in some seeds."
    )
    print(
        f"Consider stronger T_START (current={T_START}) or longer ANNEAL_EP (current={ANNEAL_EP})."
    )
else:
    print("Verdict: ❌ UNSTABLE — not all seeds converge.")
    print(
        "Debug: check attention evolution plots — which seed fails and at what epoch?"
    )